In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys
import jax 
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)

import matplotlib.pyplot as plt
import numpy as np
sys.path.append(os.path.realpath(".."))

from dynamax.linear_gaussian_ssm import LinearGaussianSSM, ParamsLGSSM

import hippocampalseq as hse
import hippocampalseq.preprocessing as hsep
import hippocampalseq.models as hsem
import hippocampalseq.utils as hseu
import hippocampalseq.plotting as hsepl

In [ ]:
theta_time_window_ms = 60#10#250
theta_time_window_s  = theta_time_window_ms / 1000
theta_time_window_advance_ms = 60#5#250
theta_time_window_advance_s  = theta_time_window_advance_ms / 1000

ripple_time_window_ms = 3.0#5.0
ripple_time_window_s  = ripple_time_window_ms / 1000
ripple_time_window_advance_ms = 3.0#5.0
ripple_time_window_advance_s  = ripple_time_window_advance_ms / 1000


bin_size = 2
data_path = os.path.realpath("../data")
rat_name = "Janni"
session = 1
track_type = "Open"
results_path = f"../results/{rat_name}/{track_type}{session}"

In [ ]:
(
    raw_data,
    place_field_data,
    theta_data,
    ripple_data
) = hse.load_and_preprocess(
    data_path,
    rat_name,
    session,

    track_type=track_type,
    place_field_posterior=True,
    environment_size=(0,0,200,200),
    velocity_cutoff=5.0,
    bin_size_cm=bin_size,
    theta_time_window_ms=theta_time_window_ms,
    theta_time_window_advance_ms=theta_time_window_advance_ms,
    ripple_time_window_ms=ripple_time_window_ms,
    ripple_time_window_advance_ms=ripple_time_window_advance_ms
)

In [ ]:
class Momentum(LinearGaussianSSM):
    def __init__(
        self,
        place_fields,
        spikemats,
        dt,
        environment_size,
        bin_size,
        seed=None
    ):
        super().__init__(4, 2, 0, False, False)

        if seed is None:
            seed = int.from_bytes(os.urandom(4))
        seed = jax.random.key(seed)

        self.dt = dt
        self.environment_size = environment_size
        self.bin_size = bin_size

        self.grid = hseu.create_grid(self.environment_size, self.bin_size)
        place_fields = jnp.array(place_fields)

        self.emission_probabilities 

        @jax.jit
        def approximate(spikemat, place_fields, dt, grid, bin_size):
            spikemat = jnp.array(spikemat, dtype=jnp.float64)
            emission_probabilities = calc_poisson_emission_probabilities_2d(
                spikemat, 
                place_fields, 
                dt
            )
            emission_probabilities /= jnp.sum(emission_probabilities, axis=1, keepdims=True)
            approx_mean,approx_cov = hseu.analytical_gaussian_approximation(
                grid,
                emission_probabilities,
                bin_size
            )
            return emission_probabilities, approx_mean, approx_cov
        approximations = jax.vmap([approximate(spikemat, place_fields, dt, grid, bin_size) for spikemat in spikemats])
        emission_probabilities, approx_mean, approx_cov = zip(*approximations)

    def _construct_transition_mat(self, decay):
        I = jnp.eye(self.state_dim, dtype=jnp.float64)
        Z = jnp.zeros((self.emission_dim, self.emission_dim), dtype=jnp.float64)

        T1 = (-decay * self.dt) * jnp.eye(self.emission_dim, dtype=jnp.float64)
        top = jnp.concat((T1, Z), axis=1)
        bottom = jnp.concat((I, Z), axis=1)
        F = jnp.concat((top, bottom), axis=0) + I
        return F

    def _construct_transition_cov(self, diffusion):
        q = diffusion * self.dt * jnp.eye(self.emission_dim, dtype=jnp.float64)
        Z = jnp.zeros((self.emission_dim, self.emission_dim), dtype=jnp.float64)
        top = jnp.concat((q, Z), axis=1)
        bottom = jnp.concat((Z, Z), axis=1)
        Q = jnp.concat((top, bottom), axis=0)
        return Q

    def initialize(
        self,
        decay = None,
        diffusion = None
    ):
        default = lambda x, x0: x if x is not None else x0

        _decay = self.random(1)
        _diffusion = self.random(1)
        self.decay = default(decay, _decay)
        self.diffusion = default(diffusion, _diffusion)
        F = self._construct_transition_mat(self.decay)
        Q = self._construct_transition_cov(self.diffusion)

        super().inititalize(
            self.key, 
            dynamics_weights=F,
            dynamics_noise_covariance=Q
        )


    def random(self, shape: tuple|int, random_type: str = 'uniform', dtype=jnp.float64, *args, **kwargs):
        nkey,skey = jax.random.split(self.key)
        if random_type == 'uniform':
            rv = jax.random.uniform(skey, shape=shape, dtype=dtype, **kwargs)
        elif random_type == 'normal':
            rv = jax.random.normal(skey, shape=shape, dtype=dtype, **kwargs)
        else:
            raise Exception(f"{random_type} is not a supported RNG type")
        self.key = nkey
        return rv